# Read in Overall Cohort

(prior to filtering for contiguous 10-day window of data)

In [3]:
import os
import pandas as pd
my_bucket = os.getenv('WORKSPACE_BUCKET')
prefix = f'{my_bucket}/data/dfs/'
daily_df = pd.read_csv(f"{prefix}daily_df_v2_prepped.csv")
labels = pd.read_csv(f"{prefix}daily_df_labels_v2.csv", index_col=0)

In [5]:
daily_df.columns

Index(['person_id', 'date', 'minute_in_bed', 'minute_asleep',
       'minute_after_wakeup', 'minute_awake', 'activity_calories',
       'calories_bmr', 'calories_out', 'fairly_active_minutes', 'floors',
       'lightly_active_minutes', 'marginal_calories', 'sedentary_minutes',
       'steps', 'very_active_minutes', 'mean_hr', 'median_hr', 'std_hr',
       'min_hr', 'max_hr', 'range_hr', 'count_hr', 'proportion_resting',
       'proportion_moderate', 'proportion_high', 'morning_hr', 'afternoon_hr',
       'evening_hr', 'night_hr'],
      dtype='object')

In [8]:
daily_df.person_id.unique().shape

(13779,)

In [12]:
labels.reset_index().columns

Index(['person_id', 'Anxiety disorder', 'Depressive disorder', 'No disorder'], dtype='object')

----

----

# Compare Physiological Features Across Diagnostic Groups

In [70]:
# Step 1: Calculate mean for each person
person_means = daily_df.groupby('person_id').agg({
    'minute_asleep': 'mean',
    'mean_hr': 'mean',
    'steps': 'mean'
}).reset_index()

# Step 2: Merge with labels
person_means_labeled = person_means.merge(labels.reset_index(), on='person_id', how='inner')

# Step 3: Create group labels
def assign_group(row):
    if row['Anxiety disorder'] == 1 and row['Depressive disorder'] == 1:
        return 'Anxiety AND Depression'
    elif row['Anxiety disorder'] == 1:
        return 'Anxiety Only'
    elif row['Depressive disorder'] == 1:
        return 'Depression Only'
    elif row['No disorder'] == 1:
        return 'No Disorder'
    else:
        return 'Unknown'

person_means_labeled['group'] = person_means_labeled.apply(assign_group, axis=1)

# Step 4: Calculate group averages
group_averages = person_means_labeled.groupby('group').agg({
    'minute_asleep': 'mean',
    'mean_hr': 'mean',
    'steps': 'mean',
    'person_id': 'count'  # Also count how many people in each group
}).rename(columns={'person_id': 'n_people'})

print(group_averages)

                        minute_asleep    mean_hr        steps  n_people
group                                                                  
Anxiety AND Depression     373.388389  77.912296  6927.479017      1460
Anxiety Only               377.924236  76.578179  8006.818669       826
Depression Only            371.704824  75.666614  8043.739391       701
No Disorder                372.752002  75.712114  7963.770936     10792


In [71]:
# Create summary table with Standard Deviation
def format_mean_sd(group_data, col):
    mean = group_data[col].mean()
    sd = group_data[col].std()
    return f"{mean:.1f} ({sd:.1f})"

summary_table = pd.DataFrame()

for group_name in ['No Disorder', 'Anxiety Only', 'Depression Only', 'Anxiety AND Depression']:
    group_data = person_means_labeled[person_means_labeled['group'] == group_name]
    
    summary_table.loc[group_name, 'N'] = len(group_data)
    summary_table.loc[group_name, 'Minutes Asleep - Mean (SD)'] = format_mean_sd(group_data, 'minute_asleep')
    summary_table.loc[group_name, 'Heart Rate - Mean (SD)'] = format_mean_sd(group_data, 'mean_hr')
    summary_table.loc[group_name, 'Steps - Mean (SD)'] = format_mean_sd(group_data, 'steps')

summary_table

,N,Minutes Asleep - Mean (SD),Heart Rate - Mean (SD),Steps - Mean (SD)
No Disorder,10792.0,372.8 (77.0),75.7 (8.3),7963.8 (3606.9)
Anxiety Only,826.0,377.9 (78.2),76.6 (7.9),8006.8 (3288.4)
Depression Only,701.0,371.7 (82.1),75.7 (7.9),8043.7 (21856.1)
Anxiety AND Depression,1460.0,373.4 (79.9),77.9 (8.6),6927.5 (6692.2)


----

----

# Get Condition Codes

(only contains codes falling under AoU defined Anxiety Disorder or Depressive Disorder)

In [90]:
# Get unique person IDs as a list
person_ids = daily_df.person_id.unique().tolist()

condition_sql = f"""
    SELECT DISTINCT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary 
    FROM
        `{os.environ["WORKSPACE_CDR"]}.condition_occurrence` c_occurrence 
    LEFT JOIN
        `{os.environ["WORKSPACE_CDR"]}.concept` c_standard_concept 
            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id
    WHERE
        c_occurrence.person_id IN UNNEST({person_ids})
        AND c_occurrence.condition_concept_id IN (
            SELECT DISTINCT c.concept_id 
            FROM `{os.environ["WORKSPACE_CDR"]}.cb_criteria` c 
            JOIN (
                SELECT CAST(cr.id as string) AS id       
                FROM `{os.environ["WORKSPACE_CDR"]}.cb_criteria` cr       
                WHERE concept_id IN (440383, 442077)       
                AND full_text LIKE '%_rank1]%'
            ) a 
            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                OR c.path LIKE CONCAT('%.', a.id) 
                OR c.path LIKE CONCAT(a.id, '.%') 
                OR c.path = a.id) 
            WHERE is_standard = 1 AND is_selectable = 1
        )
"""

condition_df = pd.read_gbq(
    condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

Downloading:   0%|          | 0/9776 [00:00<?, ?rows/s]

In [91]:
condition_df.person_id.unique().shape

(2987,)

## Get Code Counts

(Note: 2,987/13,779 have an anxiety/depressive disorder)

In [92]:
# Group by the concept information and count occurrences
code_counts = condition_df.groupby([
    'standard_concept_name', 
    'standard_concept_code', 
    'standard_vocabulary'
]).size().reset_index(name='count')

# Print as CSV format
print(code_counts.to_csv(index=False))

standard_concept_name,standard_concept_code,standard_vocabulary,count
Acute depression,712823008,SNOMED,1
Acute post-trauma stress state,192042008,SNOMED,3
Acute situational disturbance,192041001,SNOMED,8
Acute stress disorder,67195008,SNOMED,291
Agoraphobia,70691001,SNOMED,5
Agoraphobia without history of panic disorder,61569007,SNOMED,1
Alcohol-induced anxiety disorder,34938008,SNOMED,1
Anxiety disorder,197480006,SNOMED,1781
Anxiety disorder due to a general medical condition,52910006,SNOMED,3
Anxiety disorder of adolescence,37868008,SNOMED,1
Anxiety disorder of childhood OR adolescence,109006,SNOMED,3
Atypical depressive disorder,191659001,SNOMED,1
"Bipolar affective disorder, current episode depression",191627008,SNOMED,59
"Bipolar affective disorder, currently depressed, in full remission",191634005,SNOMED,6
"Bipolar affective disorder, currently depressed, mild",191629006,SNOMED,14
"Bipolar affective disorder, currently depressed, moderate",191630001,SNOMED,25
Chronic depression,

## Export Codes Associated with Anonymized `person_id`

(to enable future research directions, such as comorbidity analysis)

In [94]:
# Create a mapping from original person_id to new anonymized IDs
unique_persons = condition_df['person_id'].unique()
person_id_mapping = {old_id: new_id for new_id, old_id in enumerate(unique_persons, start=1)}

anonymized_condition_df = condition_df.copy()
anonymized_condition_df['person_id'] = anonymized_condition_df['person_id'].map(person_id_mapping)

anonymized_condition_df.to_csv(index=False)

'person_id,condition_concept_id,standard_concept_name,standard_concept_code,standard_vocabulary\n1,439253,"Bipolar affective disorder, currently depressed, mild",191629006,SNOMED\n2,437528,"Bipolar affective disorder, currently depressed, moderate",191630001,SNOMED\n3,437528,"Bipolar affective disorder, currently depressed, moderate",191630001,SNOMED\n4,4147466,Panic disorder with agoraphobia,35607004,SNOMED\n5,4269493,Major depression in full remission,63412003,SNOMED\n6,432285,Recurrent major depressive episodes,268621008,SNOMED\n7,432285,Recurrent major depressive episodes,268621008,SNOMED\n8,4242733,Premenstrual dysphoric disorder,596004,SNOMED\n9,43531624,Severe recurrent major depression,281000119103,SNOMED\n10,4039212,Separation anxiety disorder of childhood,11806006,SNOMED\n11,436074,Panic disorder,371631005,SNOMED\n12,436074,Panic disorder,371631005,SNOMED\n13,436074,Panic disorder,371631005,SNOMED\n14,4176002,Major depression in remission,42810003,SNOMED\n15,381537,Organic an

----

----

# Dummy Classifier

(always predicts majority class)

In [20]:
"""
Dummy Classifier Baseline for All of Us Conformal Prediction Experiments

This script generates baseline metrics from a dummy classifier that always 
predicts the majority class ('No disorder') for comparison with LSTM and 
conformal prediction methods.

Uses Hamming Accuracy (label-based accuracy) as the primary metric:
  Hamming Accuracy = 1 - Hamming Loss
                   = Average fraction of labels correctly predicted
                   = (1 / (n_samples × n_labels)) × Σ I(y_true == y_pred)

This matches the 'base_contains_true_label_percent' metric (64.2% avg) from
the doesexperiment outputs (CSV output cell saved to file):
'standard_conformal_prediction_experiment_results__multilabel.csv'
from 
'__3__run_inductive-conformal-prediction_experiments__cleared_output.ipynb'.

Run this after the main experiments to establish baseline comparisons.
"""

import os
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
from google.cloud import storage
import io

# Configuration
my_bucket = os.getenv('WORKSPACE_BUCKET')
prefix = f'{my_bucket}/data/experiments/'
num_runs = 100
num_days = 10


def load_y_test(run_num, num_days, with_demographics):
    """Load y_test array for a specific run from GCS."""
    suffix = f'_{num_days}_days_{"with" if with_demographics else "wout"}_demo.npy'
    bucket_name = os.getenv('WORKSPACE_BUCKET').replace('gs://', '')
    blob_path = f'data/experiments/{run_num}/y_test{suffix}'
    
    # Initialize GCS client
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_path)
    
    # Check if blob exists
    if not blob.exists():
        raise FileNotFoundError(f"Blob does not exist: gs://{bucket_name}/{blob_path}")
    
    # Download as bytes and load with numpy
    bytes_data = blob.download_as_bytes()
    return np.load(io.BytesIO(bytes_data))


def dummy_predictions_multilabel(y_test):
    """
    Generate dummy predictions for multi-label classification.
    Always predicts [0, 0, 1] (No Anxiety, No Depression, Has No Disorder)
    
    Args:
        y_test: Ground truth labels of shape (n_samples, 3)
    
    Returns:
        Dummy predictions of shape (n_samples, 3)
    """
    n_samples = y_test.shape[0]
    # Always predict: No Anxiety (0), No Depression (0), No Disorder (1)
    return np.tile([0, 0, 1], (n_samples, 1))


def evaluate_multilabel_dummy(runs=100, with_demographics=True):
    """
    Evaluate dummy classifier on multi-label task across all runs.
    Calculates Hamming accuracy (label-based accuracy) to match notebook CSV.
    """
    all_hamming_accuracies = []  # Hamming accuracy = 1 - Hamming loss
    all_metrics = []
    label_names = ['Anxiety', 'Depression', 'No Disorder']
    
    successful_runs = 0
    failed_runs = []
    
    for run in range(runs):
        try:
            y_test = load_y_test(run, num_days, with_demographics)
            y_pred = dummy_predictions_multilabel(y_test)
            
            # Calculate Hamming accuracy (label-based accuracy):
            # For each (sample, label) pair, check if prediction matches ground truth
            # This is: 1 - Hamming Loss
            # Matches: base_contains_true_label_percent from CSV
            total_correct = 0
            total_pairs = 0
            
            for label_idx in range(3):
                y_true_label = y_test[:, label_idx]
                y_pred_label = y_pred[:, label_idx]
                correct = (y_true_label == y_pred_label).sum()
                total_correct += correct
                total_pairs += len(y_true_label)
            
            hamming_accuracy = total_correct / total_pairs
            all_hamming_accuracies.append(hamming_accuracy)
            
            # Calculate per-label metrics
            for label_idx, label_name in enumerate(label_names):
                y_true_label = y_test[:, label_idx]
                y_pred_label = y_pred[:, label_idx]
                
                precision, recall, f1, _ = precision_recall_fscore_support(
                    y_true_label, y_pred_label, average='binary', zero_division=0
                )
                
                all_metrics.append({
                    'run': run,
                    'label': label_name,
                    'label_idx': label_idx,
                    'accuracy_per_label': accuracy_score(y_true_label, y_pred_label),
                    'precision': precision,
                    'recall': recall,
                    'f1': f1,
                    'support_true': int(y_true_label.sum()),
                    'support_false': int((~y_true_label.astype(bool)).sum())
                })
            
            successful_runs += 1
                
        except FileNotFoundError as e:
            failed_runs.append(run)
            if run < 5:  # Only print first few errors
                print(f"  Warning: y_test file not found for run {run}")
            continue
        except Exception as e:
            failed_runs.append(run)
            print(f"  Error processing run {run}: {str(e)}")
            continue
    
    print(f"\n  Successfully processed: {successful_runs}/{runs} runs")
    if failed_runs:
        if len(failed_runs) <= 10:
            print(f"  Failed runs: {failed_runs}")
        else:
            print(f"  Failed runs: {failed_runs[:10]}... and {len(failed_runs)-10} more")
    
    return pd.DataFrame(all_metrics), all_hamming_accuracies


def summarize_results(metrics_df, hamming_accuracies, task_type):
    """
    Print summary statistics for dummy classifier performance.
    Uses Hamming accuracy (label-based accuracy) to match notebook CSV.
    """
    print(f"\n{'='*80}")
    print(f"DUMMY CLASSIFIER RESULTS - {task_type.upper()}")
    print(f"{'='*80}\n")
    
    if len(hamming_accuracies) == 0:
        print("ERROR: No successful runs to analyze!")
        return None
    
    print(f"Hamming Accuracy (Label-Based Accuracy) across {len(hamming_accuracies)} runs:")
    print(f"  (= 1 - Hamming Loss)")
    print(f"  (Matches 'base_contains_true_label_percent' from CSV)")
    print(f"  Mean: {np.mean(hamming_accuracies)*100:.2f}%")
    print(f"  Median: {np.median(hamming_accuracies)*100:.2f}%")
    print(f"  Std: {np.std(hamming_accuracies)*100:.2f}%")
    print(f"  Min: {np.min(hamming_accuracies)*100:.2f}%")
    print(f"  Max: {np.max(hamming_accuracies)*100:.2f}%")
    
    if metrics_df.empty:
        print("\nERROR: No metrics to summarize!")
        return None
    
    print("\n" + "-"*80)
    print("Per-Label Metrics:")
    print("(Dummy classifier having high per-label accuracy indicates class imbalance)")
    print("-"*80)
    
    summary = metrics_df.groupby('label').agg({
        'accuracy_per_label': ['mean', 'std'],
        'precision': ['mean', 'std'],
        'recall': ['mean', 'std'],
        'f1': ['mean', 'std']
    }).round(4)
    
    print(summary)
    
    # Verify: Hamming accuracy should equal average of per-label accuracies
    avg_per_label_acc = metrics_df.groupby('label')['accuracy_per_label'].mean().mean()
    print(f"\n  Average of per-label accuracies: {avg_per_label_acc*100:.2f}%")
    print(f"  (Should equal Hamming accuracy above)")
    
    return summary


def compare_to_baseline(dummy_hamming_acc, lstm_hamming_acc=0.6420):
    """
    Compare dummy classifier to LSTM baseline using Hamming accuracy.
    """
    print(f"\n{'='*80}")
    print("COMPARISON TO LSTM BASELINE (Hamming Accuracy)")
    print(f"{'='*80}\n")
    print(f"Dummy Classifier Mean Hamming Accuracy: {dummy_hamming_acc*100:.2f}%")
    print(f"LSTM Mean Hamming Accuracy (from CSV): {lstm_hamming_acc*100:.2f}%")
    print(f"LSTM Improvement over Dummy: {(lstm_hamming_acc - dummy_hamming_acc)*100:.2f} percentage points")
    
    if dummy_hamming_acc > 0:
        print(f"Relative Change: {((lstm_hamming_acc - dummy_hamming_acc) / dummy_hamming_acc)*100:.2f}%")
    
    if lstm_hamming_acc < dummy_hamming_acc:
        print(f"\n  NOTE: LSTM has LOWER Hamming accuracy than dummy classifier!")
        print(f"    This indicates:")
        print(f"    - The dummy exploits class imbalance (always predicts majority class)")
        print(f"    - LSTM tries to detect actual patterns (harder but more useful)")
        print(f"    - Simply predicting majority class gives higher raw accuracy")


# Main execution
print("="*80)
print("Generating Dummy Classifier Baseline Results")
print("="*80)

# Multi-label evaluation (with demographics)
print("\n\nEvaluating Multi-Label Classification (WITH Demographics)...")
ml_metrics_with, ml_acc_with = evaluate_multilabel_dummy(runs=100, with_demographics=True)

if len(ml_acc_with) > 0:
    ml_summary_with = summarize_results(ml_metrics_with, ml_acc_with, 'multi-label')
else:
    print("\nERROR: No successful runs for multi-label WITH demographics")
    ml_summary_with = None

# Comparisons to LSTM baseline
if len(ml_acc_with) > 0:
    print("\n\nMulti-Label Comparisons (WITH Demographics):")
    print("Using Hamming Accuracy (label-based accuracy = 1 - Hamming loss)")
    print("Matches CSV 'base_contains_true_label_percent'")
    compare_to_baseline(np.mean(ml_acc_with), lstm_hamming_acc=0.6420)
    
    
# Multi-label evaluation (without demographics)
print("\n\nEvaluating Multi-Label Classification (WITHOUT Demographics)...")
ml_metrics_wo, ml_acc_wo = evaluate_multilabel_dummy(runs=100, with_demographics=False)

if len(ml_acc_wo) > 0:
    ml_summary_wo = summarize_results(ml_metrics_wo, ml_acc_wo, 'multi-label')
else:
    print("\nERROR: No successful runs for multi-label WITHOUT demographics")
    ml_summary_wo = None

# Comparisons to LSTM baseline
if len(ml_acc_wo) > 0:
    print("\n\nMulti-Label Comparisons (WITHOUT Demographics):")
    print("Using Hamming Accuracy (label-based accuracy = 1 - Hamming loss)")
    print("Matches CSV 'base_contains_true_label_percent'")
    compare_to_baseline(np.mean(ml_acc_wo), lstm_hamming_acc=0.6420)

print("\n\n" + "="*80)
print("SUMMARY")
print("="*80)
if ml_summary_with is not None:
    print("Multi-label WITH demographics: SUCCESS")
    print(f"  - Hamming Accuracy: {np.mean(ml_acc_with)*100:.2f}%")
if ml_summary_wo is not None:
    print("Multi-label WITHOUT demographics: SUCCESS")
    print(f"  - Hamming Accuracy: {np.mean(ml_acc_wo)*100:.2f}%")
print("="*80)

Generating Dummy Classifier Baseline Results


Evaluating Multi-Label Classification (WITH Demographics)...

  Successfully processed: 100/100 runs

DUMMY CLASSIFIER RESULTS - MULTI-LABEL

Hamming Accuracy (Label-Based Accuracy) across 100 runs:
  (= 1 - Hamming Loss)
  (Matches 'base_contains_true_label_percent' from CSV)
  Mean: 82.08%
  Median: 82.07%
  Std: 0.47%
  Min: 81.13%
  Max: 83.15%

--------------------------------------------------------------------------------
Per-Label Metrics:
(Dummy classifier having high per-label accuracy indicates class imbalance)
--------------------------------------------------------------------------------
            accuracy_per_label         precision         recall          f1  \
                          mean     std      mean     std   mean  std   mean   
label                                                                         
Anxiety                 0.8349  0.0048    0.0000  0.0000    0.0  0.0  0.000   
Depression              0.84

----

----

----

----

----

----

---